# YUV ↔ RGB Conversion Matrices

This notebook computes **single‑precision (`float32`) 3 × 3 matrices** (and accompanying offset vectors) for converting:

* **YUV (limited range 16 – 235/240)** → **RGB (full range 0 – 255)**
* **RGB (full range 0 – 255)** → **YUV (limited range 16 – 235/240)**

for the following color spaces:

| Standard | \(K_R\) | \(K_B\) |
|----------|---------|---------|
| BT.601   | 0.2990  | 0.1140  |
| BT.709   | 0.2126  | 0.0722  |
| BT.2020  | 0.2627  | 0.0593  |

The formulas follow ITU‑R recommendations and scale factors defined for limited‑range (video‑range) signals.

In [1]:
import numpy as np

def compute_yuv_to_rgb_matrix(kr: float, kb: float) -> np.ndarray:
    """Return a 3×3 float32 matrix converting YUV (16‑235/240)→RGB (0‑255)."""
    kg = 1.0 - kr - kb
    alpha = 255.0 / 219.0      # Luma scaling (Y)
    beta  = 255.0 / 224.0      # Chroma scaling (Cb/Cr)

    m = np.empty((3, 3), dtype=np.float32)
    # R row
    m[0, 0] = alpha
    m[0, 1] = 0.0
    m[0, 2] = beta * 2.0 * (1.0 - kr)
    # G row
    m[1, 0] = alpha
    m[1, 1] = -beta * 2.0 * kb * (1.0 - kb) / kg
    m[1, 2] = -beta * 2.0 * kr * (1.0 - kr) / kg
    # B row
    m[2, 0] = alpha
    m[2, 1] = beta * 2.0 * (1.0 - kb)
    m[2, 2] = 0.0
    return m

def compute_rgb_to_yuv_matrix(kr: float, kb: float) -> np.ndarray:
    """Return a 3×3 float32 matrix converting RGB (0‑255)→YUV (16‑235/240)."""
    kg = 1.0 - kr - kb
    # Scaling factors for limited range
    scale_y  = 219.0 / 255.0
    scale_cb = 224.0 / (2.0 * (1.0 - kb) * 255.0)
    scale_cr = 224.0 / (2.0 * (1.0 - kr) * 255.0)

    m = np.empty((3, 3), dtype=np.float32)
    # Y' (luma)
    m[0] = [kr * scale_y, kg * scale_y, kb * scale_y]
    # Cb (blue‑difference)
    m[1] = [-kr * scale_cb, -kg * scale_cb,  (1.0 - kb) * scale_cb]
    # Cr (red‑difference)
    m[2] = [(1.0 - kr) * scale_cr, -kg * scale_cr, -kb * scale_cr]
    return m

def print_matrices(name: str, kr: float, kb: float) -> None:
    y2r = compute_yuv_to_rgb_matrix(kr, kb)
    r2y = compute_rgb_to_yuv_matrix(kr, kb)
    print(f"\n=== {name} ===")
    print("YUV → RGB matrix:")
    print(y2r)
    print("\nRGB → YUV matrix:")
    print(r2y)


In [2]:
standards = {
    "BT.601":  (0.2990, 0.1140),
    "BT.709":  (0.2126, 0.0722),
    "BT.2020": (0.2627, 0.0593),
}

for name, (kr, kb) in standards.items():
    print_matrices(name, kr, kb)


=== BT.601 ===
YUV → RGB matrix:
[[ 1.1643835   0.          1.5960268 ]
 [ 1.1643835  -0.3917623  -0.81296766]
 [ 1.1643835   2.0172322   0.        ]]

RGB → YUV matrix:
[[ 0.25678822  0.5041294   0.09790588]
 [-0.1482229  -0.2909928   0.4392157 ]
 [ 0.4392157  -0.3677883  -0.07142738]]

=== BT.709 ===
YUV → RGB matrix:
[[ 1.1643835   0.          1.7927411 ]
 [ 1.1643835  -0.21324861 -0.53290933]
 [ 1.1643835   2.1124017   0.        ]]

RGB → YUV matrix:
[[ 0.18258588  0.6142306   0.06200706]
 [-0.10064373 -0.33857197  0.4392157 ]
 [ 0.4392157  -0.39894217 -0.04027352]]

=== BT.2020 ===
YUV → RGB matrix:
[[ 1.1643835  0.         1.6786741]
 [ 1.1643835 -0.1873261 -0.6504243]
 [ 1.1643835  2.1417723  0.       ]]

RGB → YUV matrix:
[[ 0.22561294  0.58228236  0.05092824]
 [-0.12265543 -0.31656027  0.4392157 ]
 [ 0.4392157  -0.4038902  -0.0353255 ]]
